In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.special import factorial

#######
# fig 1, comparing the k space operators, k_E, k_B, k1 and the dispersion relation omega
# fig 2, comparing the error in the phase velocity for 8, 12 and 16 coefficients
# in fig 3, Comparing the cancellation factor (indicates E B balance) for the yee, xu and new solver from Li 2020
# 'A new field solver for modeling of relativistic particle-laser interactions using the particle-in-cell algorithm'
#######


# function kernel_sine( x, arglst ) in fortran line 1235. Not sure about the use of v and u, looks related to a fourier
# expansion to approximate the operator. Basically copied the fortran code
def kernel_sine(x, args):
    # args
    u = (2.0 * args[0] - 1.0) * np.pi
    v = (2.0 * args[1] - 1.0) * np.pi
    return np.sin(u * x) * np.sin(v * x)


# function kernel_dual_e( x, arglst ) fortran line 1302
# paper equation 14
def kernel_dual_e(x, args):
    u = (2.0 * args[0] - 1.0) * np.pi
    v = args[1] * np.pi
    k_cons = np.sin(v * x) / v
    omega = np.arcsin(v * k_cons) / v
    return np.sin(u * x) * k_cons / np.cos(v * omega)


# function kernel_dual_b( x, arglst ) fortran line 1321
# paper equation 15
def kernel_dual_b(x, args):
    u = (2.0 * args[0] - 1.0) * np.pi
    v = args[1] * np.pi
    k_cons = np.sin(v * x) / v
    omega = np.arcsin(v * k_cons) / v

    return np.sin(u * x) * k_cons * np.cos(v * omega)


# function weight( x, w, n ) fortran line 1695
# weighting function from appendix b
def weight(x, w, n):
    return np.exp(-np.log(2) * (x / w) ** n)


def integral(fun, args, upper=0.5, lower=0.0, tol=1e-16, w=0.36, n=10):
    def integrand(x, *in_args):
        # Unpack the arguments
        true_args = in_args[0]
        w = in_args[1]
        n = in_args[2]
        return fun(x, true_args) * weight(x, w, n)

    result, error = quad(integrand, lower, upper, args=(args, w, n), epsabs=1e-16)
    return result


# subroutine precond( A, b, n ) line 1622
# from fortran comment: precondition the linear equations Ax=b of dimension n to improve the condition number
def precondition(A, b):
    d = 1.0 / np.max(np.abs(A), axis=1)
    A_scaled = A * d[:, np.newaxis]
    b_scaled = b * d
    return A_scaled, b_scaled


# subroutine gen_solver_coef( fun, arglst, ord, n_coef, coef, w, n ) implemented from fortran 1162
# perform the integral from B.8 and B.9, build the A and b matrices,
def gen_solver_coef(kernel_func, args, ord, n_coef, w=0.35, n=10):
    size = n_coef + ord // 2
    A = np.zeros((size, size))
    b = np.zeros(size)

    for i in range(n_coef):
        for j in range(n_coef):
            lst = [i + 1, j + 1]
            A[i, j] = (2 / np.pi**2) * integral(kernel_sine, lst, w=w, n=n)

    for i in range(ord // 2):
        for j in range(n_coef):
            val = (2 * j + 1) ** (2 * i + 1) / factorial(2 * i + 1)
            A[i + n_coef, j] = val
            A[j, i + n_coef] = val

    for i in range(n_coef):
        lst = [i + 1] + list(args)
        b[i] = (2 / np.pi) * integral(kernel_func, lst, w=w, n=n)
    b[n_coef] = 1.0

    # Print A and b for debugging
    # print("Matrix A (before preconditioning):\n", A)
    # print("Vector b (before preconditioning):\n", b)

    A_scaled, b_scaled = precondition(A, b)

    # With this higher precision approach:
    from scipy.linalg import qr, solve_triangular

    q, r = qr(A_scaled, mode="economic")
    y = np.dot(q.T, b_scaled)
    x = solve_triangular(r, y)

    return x[:n_coef]


# takes the calculated stencil coefficients and does equation 6 from the paper
def calculate_operators(k1, dx1, dt, coef_e, coef_b):
    """Calculate kE1 and kB1 operators"""
    kE1 = np.zeros_like(k1)
    kB1 = np.zeros_like(k1)

    for j in range(len(coef_e)):
        kE1 += coef_e[j] * np.sin((2 * j + 1) * k1 * dx1 / 2) / (dx1 / 2)
        kB1 += coef_b[j] * np.sin((2 * j + 1) * k1 * dx1 / 2) / (dx1 / 2)

    return kE1, kB1


def calculate_phase_velocity(kE1, kB1, k1, dt):
    """Calculate phase velocity using equation 5 from the paper"""
    # βφ ≡ ω/k1 = (2/k1Δt) arcsin((Δt/2)√[k]B1[k]E1)
    return (2 / (k1 * dt)) * np.arcsin((dt / 2) * np.sqrt(kE1 * kB1))


# cancellation factor defined in figure 2 description. Ideally F = 1
def calculate_cancellation_factor(kE1, kB1, k1, dt):
    """Calculate cancellation factor F with proper normalization"""
    k1t = np.sin(k1 * dt / 2) / (dt / 2)

    kE1_scaled = kE1 / k1t
    kB1_scaled = kB1 / k1t

    omega = k1 * np.sqrt(kE1_scaled * kB1_scaled)

    F = np.sqrt(kE1_scaled / kB1_scaled) * np.cos(omega * dt / 2)

    return F


# used claude to write this part to make figure 2 from the paper
def test_operators():
    """test and plot results"""
    # Parameters matching paper

    k0 = 1.0
    dx1 = 0.2 / k0
    dt = 0.5 * dx1
    kg1 = 2 * np.pi / dx1

    k1 = np.linspace(1e-10, 0.5 * kg1, 2000)

    # Create figure
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # Plot operators for different stencil sizes
    for M, color in zip([8, 12, 16], ["gold", "red", "blue"]):
        coef_e = gen_solver_coef(kernel_dual_e, [dt / dx1], 2, M)
        coef_b = gen_solver_coef(kernel_dual_b, [dt / dx1], 2, M)
        print(f"E Stencil coefficients for M={M}:", coef_e)
        print(f"B Stencil coefficients for M={M}:", coef_b)
        print(coef_b * coef_e)

        # Calculate operators
        kE1, kB1 = calculate_operators(k1, dx1, dt, coef_e, coef_b)

        # Plot operators (only for M=16)
        if M == 16:
            omega_t = (2 / dt) * np.arcsin((dt / 2) * np.sqrt(kE1 * kB1))
            axs[0].plot(k1 / kg1, kE1 / kg1, "b-", label="[k]E1/kg1")
            axs[0].plot(k1 / kg1, kB1 / kg1, "r-", label="[k]B1/kg1")
            axs[0].plot(k1 / kg1, omega_t / kg1, "y-", label="ω/kg1")
            axs[0].plot(k1 / kg1, k1 / kg1, "k--", label="k1/kg1")

        # Phase velocity error
        vph = calculate_phase_velocity(kE1, kB1, k1, dt)

        axs[1].semilogy(k1 / kg1, np.abs(vph - 1), color=color, label=f"{M} coef.")

        # Cancellation factor
        F = calculate_cancellation_factor(kE1, kB1, k1, dt)
        axs[2].plot(k1 / kg1, F, color=color, label=f"{M} coef.")

    # Add Yee solver comparison
    k1_yee = np.sin(k1 * dx1 / 2) / (dx1 / 2)
    F_yee = calculate_cancellation_factor(k1_yee, k1_yee, k1, dt)
    axs[2].plot(k1 / kg1, F_yee, "purple", label="Yee")

    # Add Xu solver comparison
    k1_xu = np.sin(k1 * dt / 2) / (dt / 2)
    F_xu = calculate_cancellation_factor(k1_xu, k1_xu, k1, dt)
    axs[2].plot(k1 / kg1, F_xu, "green", label="Xu")

    # Configure plots
    titles = ["(a) Operators", "(b) Phase velocity error", "(c) Cancellation factor"]
    for ax, title in zip(axs, titles):
        ax.set_title(title)
        ax.set_xlabel("k1/kg1")
        ax.grid(True, which="both", alpha=0.3)
        ax.set_xlim(0, 0.5)

    axs[0].set_ylabel("Normalized magnitude")
    axs[0].set_ylim(0, 0.5)

    axs[1].set_ylabel("|βφ - 1|")
    axs[1].set_yscale("log")
    axs[1].set_ylim(1e-15, 1)

    axs[2].set_ylabel("Factor F")
    axs[2].set_ylim(0.7, 1.05)

    for ax in axs:
        ax.legend(loc="best")

    plt.tight_layout()
    return fig, axs


fig, axs = test_operators()
plt.show()